# Inflation différenciée en France: tableau comparatif 2026

Ce notebook reprend la logique du mémoire en appliquant les prix de l'Insee d'août 2026 aux paniers de consommation de Budget de famille 2017.

Les résultats sont des calculs à paniers fixes, et non des indices catégoriels publiés par l'Insee.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.analyse import CATEGORY_SPECS, build_comparison_table, calculate_category, load_price_ratios

divisions, ratios, metadata = load_price_ratios()
all_results = []
for category, spec in CATEGORY_SPECS.items():
    result, _ = calculate_category(category, spec, ratios)
    all_results.append(result)
results = pd.concat(all_results, ignore_index=True)
comparison = build_comparison_table(results)

## Toutes les catégories

**Définition des déciles:** ils partagent la distribution des niveaux de vie en dix groupes de même taille, classés du plus faible au plus élevé. Ici, le décile 1 correspond aux 10 % situés en bas de la distribution et le décile 10 aux 10 % situés en haut.

In [ ]:
tableau = comparison[[
    'dimension', 'group_label', 'modeled_inflation',
    'difference_vs_modeled_total', 'rank_within_dimension'
]].copy()
tableau.columns = [
    'Dimension', 'Catégorie', 'Inflation modélisée (%)',
    'Écart au panier moyen (point)', 'Rang dans la dimension'
]
tableau.style.format({
    'Inflation modélisée (%)': '{:.2f}',
    'Écart au panier moyen (point)': '{:+.2f}',
})

## Comparaison dimension par dimension

In [ ]:
for dimension, sous_tableau in tableau.groupby('Dimension', sort=False):
    print(f'\n{dimension}')
    display(sous_tableau.drop(columns='Dimension').reset_index(drop=True).style.format({
        'Inflation modélisée (%)': '{:.2f}',
        'Écart au panier moyen (point)': '{:+.2f}',
    }))

## Repères

In [ ]:
pd.Series(metadata, name='Valeur').to_frame()

Les dimensions se recoupent: un même ménage peut être rural, locataire, ouvrier et appartenir à un décile de niveau de vie. Les lignes ne doivent donc pas être additionnées entre elles. Le panier moyen modélisé ne reproduit pas exactement l'IPC officiel, car il conserve les structures de dépenses de 2017.